In [2]:
from ollama import chat
from pydantic import BaseModel, Field
from typing import List, Optional

import json

In [3]:
folder_name = "cortex_Booeshaghi2021"
img_fn = "figure.jpg"
fn = f"../../data/{folder_name}/{img_fn}"
rev_ctmap = f"../../data/{folder_name}/ctmap/rev_ctmap.json"

with open(rev_ctmap, 'r') as f:
    rev = json.load(f)

In [4]:
rev

{'L5/6 NP': 'L5/6 NP',
 'L5 PT': 'L5 PT',
 'L2/3 IT': 'L2/3 IT',
 'L6B': 'L6B',
 'L6 IT': 'L6 IT',
 'L6 CT': 'L6 CT',
 'L6 IT CAR3': 'L6 IT CAR3',
 'LAMP5': 'LAMP5',
 'SNCG': 'SNCG',
 'VIP': 'VIP',
 'SST': 'SST',
 'PVALB': 'PVALB',
 'ENDO': 'ENDO'}

In [5]:
# Restrictive class object does not allow for missing fields
class MarkerStrict(BaseModel):
    marker_name: str = Field(
        ..., 
        description="The name of the marker (can be a gene or isoform) that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene or isoform serves as a marker."
    )
    tissue_source: str = Field(
        ..., 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: str = Field(
        ..., 
        description="The species from which this marker was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerListStrict(BaseModel):
    genes: List[MarkerStrict] 

In [6]:
# Currently, we assume that the figure is an umbrella plot of marker genes.

system_prompt = f"""
You are an expert in genomics analyzing scientific literature to extract marker isoforms for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker isoform mentioned, extract:
- The **isoform name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).
- The **tissue source** where this marker gene was identified (tissue_source).
- The **species** from which the marker gene was studied (species).

If any information is missing or ambiguous, provide the most specific details available. If none of the information is available, the field can be set to null.

If some acronyms feel unfamiliar in the image you are analyzing, you are also provided this map of cell type labels. Any acronym apart of this map, case independent, is considered a cell_type_name: {json.dumps(rev)}

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{{
    "genes": [
        {{
            "marker_gene_name": "GeneX",
            "cell_type_name": "Neuron",
            "tissue_source": "Brain",
            "species": "Homo sapiens"
        }},
        ...
    ]
}}
"""

user_content = "This given image is a violin plot of isoform(s) per cell type. Spatial isoform atlas of the MOp. The scatter plots (left) show the locations of cells (black dots) in the indicated subclass within a single representative slice of the mouse MOp as assayed by MERFISH. Right, each column corresponds to a marker gene in the MERFISH dataset and each row corresponds to a subclass (number of cells in parentheses) in which one isoform (labelled on the diagonal) was differential in the SMART-seq dataset for that subclass. This spatial isoform inference links isoform expression from the SMART-seq data with the physical locations of cells expressing that isoform from the MERFISH data. The normalized gene expression values are plotted for each subclass–gene pair in TPM. The white circles in the violin plots represent the mean and the white bars represent the s.d. Please return a list of the cells and their corresponding marker isoforms (there may be multiple)"

In [11]:
def is_valid(proposed_ctmg):
    for obj in proposed_ctmg:
        if obj[list(obj.keys())[0]].upper() not in [ct.upper() for ct in rev.keys()] or obj[list(obj.keys())[1]].upper() not in [mg.upper() for mg in rev.values()]:
            return {obj["cell_type_name"].upper(): False, obj["marker_gene_name"].upper(): False}
    return {"cell_type": True, "marker_gene": True}

In [15]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2-vision"):
    return_val = list()

    while len(return_val) == 0 or not all(list(is_valid(return_val).values())):
        incorrect_ctmgs = is_valid(return_val)
        error_msg = ""
        if (incorrect_ctmgs[list(incorrect_ctmgs.keys())[0]]):
            error_msg += f" Previously, you incorrectly extracted {list(incorrect_ctmgs.keys())[0]} as a cell type name. This is not how it appears exactly as it is in the image, so you must extract the cell type again."
        if (incorrect_ctmgs[list(incorrect_ctmgs.keys())[1]]):
            error_msg += f" Previously, you incorrectly extracted {list(incorrect_ctmgs.keys())[1]} as a marker gene name. This is not how it appears exactly as it is in the image, so you must extract the marker gene again."
        response = chat(
            messages=[
                {
                    'role': 'system',
                    'content': system_prompt,
                },
                {
                    'role': 'user',
                    'content': user_content + error_msg,
                }
            ],
            model = model,
            format = data_model.model_json_schema(),
            options={'temperature': 0},  # Make responses more deterministic

        )
        genes = data_model.model_validate_json(response.message.content)
        return_val.extend(genes.model_dump()["genes"])    
    return return_val


In [16]:
print(user_content)

This given image is a violin plot of isoform(s) per cell type. Spatial isoform atlas of the MOp. The scatter plots (left) show the locations of cells (black dots) in the indicated subclass within a single representative slice of the mouse MOp as assayed by MERFISH. Right, each column corresponds to a marker gene in the MERFISH dataset and each row corresponds to a subclass (number of cells in parentheses) in which one isoform (labelled on the diagonal) was differential in the SMART-seq dataset for that subclass. This spatial isoform inference links isoform expression from the SMART-seq data with the physical locations of cells expressing that isoform from the MERFISH data. The normalized gene expression values are plotted for each subclass–gene pair in TPM. The white circles in the violin plots represent the mean and the white bars represent the s.d. Please return a list of the cells and their corresponding marker isoforms (there may be multiple)


In [17]:
genes = extract_genes(user_content, system_prompt, MarkerListStrict)
print(json.dumps(genes, indent=4))

[
    {
        "marker_name": "VIP",
        "cell_type_name": "L2/3 IT",
        "tissue_source": "MOp",
        "species": "mouse"
    },
    {
        "marker_name": "SST",
        "cell_type_name": "L6B",
        "tissue_source": "MOp",
        "species": "mouse"
    },
    {
        "marker_name": "PVALB",
        "cell_type_name": "L5 PT",
        "tissue_source": "MOp",
        "species": "mouse"
    },
    {
        "marker_name": "VIP",
        "cell_type_name": "L6 IT CAR3",
        "tissue_source": "MOp",
        "species": "mouse"
    }
]
